# Assignment 1: Market Basket Analysis on Real World Data using Python


### Aim
#To apply the Apriori algorithm on real-world transactional data to perform Market Basket Analysis (MBA), and identify frequent itemsets along with strong association rules between products.


### Description
Market Basket Analysis is a data mining technique used by retailers to understand the purchase behavior of customers by finding relationships between items that are frequently bought together. In this experiment, a set of retail transactions (customer invoices) is used as sample data. The Apriori algorithm is applied to first find frequent itemsets based on a minimum support threshold, and then association rules are generated from these itemsets using metrics such as confidence and lift. These rules help identify patterns like "customers who buy Bread and Butter also tend to buy Milk," which can be used for tasks such as product placement, cross-selling, and recommendation systems.


In [1]:
# installing mlxtend for apriori and association rules
%pip install mlxtend pandas -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
# importing required libraries
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

c:\Users\Dhiman\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
# Using sample retail transaction data (similar to what a POS/billing system would export).
# Each inner list is one customer's basket/invoice - i.e. the items they bought together in a single trip.
# sample retail transactions - each list is one customer's basket
transactions = [
    ['Milk', 'Bread', 'Butter'],
    ['Bread', 'Butter'],
    ['Milk', 'Bread'],
    ['Bread', 'Butter', 'Eggs'],
    ['Milk', 'Eggs'],
    ['Bread', 'Butter', 'Milk', 'Eggs'],
    ['Tea', 'Sugar', 'Milk'],
    ['Tea', 'Sugar'],
    ['Coffee', 'Sugar'],
    ['Tea', 'Sugar', 'Biscuits'],
    ['Bread', 'Jam'],
    ['Bread', 'Butter', 'Jam'],
    ['Chips', 'Cola'],
    ['Chips', 'Cola', 'Chocolate'],
    ['Cola', 'Chocolate'],
    ['Bread', 'Butter', 'Milk'],
    ['Milk', 'Bread', 'Eggs', 'Butter'],
    ['Tea', 'Sugar', 'Milk', 'Biscuits'],
    ['Chips', 'Chocolate'],
    ['Bread', 'Milk'],
]

print(f"Total transactions: {len(transactions)}")

Total transactions: 20


In [4]:
# Apriori needs data in a "basket format" - one row per invoice, one column per product,
# with True/False showing whether that product was bought in that invoice or not.
# TransactionEncoder does this conversion for us automatically.

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

basket_df = pd.DataFrame(te_array, columns=te.columns_)
basket_df.head()

,Biscuits,Bread,Butter,Chips,Chocolate,Coffee,Cola,Eggs,Jam,Milk,Sugar,Tea
0,False,True,True,False,False,False,False,False,False,True,False,False
1,False,True,True,False,False,False,False,False,False,False,False,False
2,False,True,False,False,False,False,False,False,False,True,False,False
3,False,True,True,False,False,False,False,True,False,False,False,False
4,False,False,False,False,False,False,False,True,False,True,False,False


In [5]:
# items/combinations appearing in at least 25% of invoices
frequent_itemsets = apriori(basket_df, min_support=0.25, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False).reset_index(drop=True)

frequent_itemsets

,support,itemsets
0,0.50,frozenset({Bread})
1,0.45,frozenset({Milk})
2,0.35,frozenset({Butter})
3,0.35,"frozenset({Bread, Butter})"
4,0.30,"frozenset({Milk, Bread})"
5,0.25,frozenset({Sugar})


In [6]:
# Generating "if a customer buys X, they are likely to also buy Y" rules from the frequent itemsets.
# We use 'lift' as the metric - lift tells us how much MORE likely Y is bought when X is bought,
# compared to Y being bought completely at random. A lift > 1 means a real, useful relationship.
# generating rules based on lift (how strong the association is)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

# converting frozenset objects into readable comma-separated strings
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

,antecedents,consequents,support,confidence,lift
0,Bread,Butter,0.35,0.700000,2.000000
1,Butter,Bread,0.35,1.000000,2.000000
2,Milk,Bread,0.30,0.666667,1.333333
3,Bread,Milk,0.30,0.600000,1.333333


In [7]:
# Not every rule with lift > 1 is worth acting on, so we tighten the filter:
# confidence >= 60% (how often the rule holds true) and lift >= 1.2 (a decent real association)
strong_rules = rules[(rules['confidence'] >= 0.6) & (rules['lift'] >= 1.2)]
strong_rules = strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

strong_rules

,antecedents,consequents,support,confidence,lift
0,Bread,Butter,0.35,0.700000,2.000000
1,Butter,Bread,0.35,1.000000,2.000000
2,Milk,Bread,0.30,0.666667,1.333333
3,Bread,Milk,0.30,0.600000,1.333333


In [8]:
# Saving the rules so they can be attached to the report/assignment submission
strong_rules.to_csv("market_basket_rules.csv", index=False)
print("Rules saved to market_basket_rules.csv\n")

print("Key insights from the analysis:")
for _, row in strong_rules.iterrows():
    items_bought = ', '.join(list(row['antecedents']))
    items_suggested = ', '.join(list(row['consequents']))
    print(f"-> Customers who buy [{items_bought}] also tend to buy [{items_suggested}] "
          f"(confidence: {row['confidence']*100:.0f}%, lift: {row['lift']:.2f})")

Rules saved to market_basket_rules.csv

Key insights from the analysis:
-> Customers who buy [B, r, e, a, d] also tend to buy [B, u, t, t, e, r] (confidence: 70%, lift: 2.00)
-> Customers who buy [B, u, t, t, e, r] also tend to buy [B, r, e, a, d] (confidence: 100%, lift: 2.00)
-> Customers who buy [M, i, l, k] also tend to buy [B, r, e, a, d] (confidence: 67%, lift: 1.33)
-> Customers who buy [B, r, e, a, d] also tend to buy [M, i, l, k] (confidence: 60%, lift: 1.33)
